In [16]:
import pickle as pkl
import random
import numpy as np
import os
import pandas as pd

# GO analysis
from goatools.obo_parser import GODag
from goatools.associations import read_gaf

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Control flag for embedding type

In [17]:
IDH1_PATH = '/home/jienihu/sc/SLformer/experiment/inference/IDH1_PRKDC_single/'

PRMT5_PATH = '/home/jienihu/sc/SLformer/experiment/inference/PRMT5_MAT2A_test/'
idh1_res = os.listdir(IDH1_PATH)
prmt5_res = os.listdir(PRMT5_PATH)

In [18]:
idh1_scoring_matrix = np.zeros((len(idh1_res), 8)) # 8 cancers
for idx in range(len(idh1_res)):
    df = pd.read_csv(IDH1_PATH + idh1_res[idx])['avg_score']
    idh1_scoring_matrix[idx] = df.values
idh1_scoring_matrix.shape

(70, 8)

In [19]:
prmt5_scoring_matrix = np.zeros((len(prmt5_res), 7)) # 7 cancers
for idx in range(len(prmt5_res)):
    df = pd.read_csv(PRMT5_PATH + prmt5_res[idx])['avg_score']
    prmt5_scoring_matrix[idx] = df.values
prmt5_scoring_matrix.shape

(70, 7)

In [20]:
idh1_glioma = idh1_scoring_matrix[:, -1]
print(idh1_glioma)
print(np.argmax(idh1_glioma))
print(idh1_glioma[np.argmax(idh1_glioma)])
idh1_glioma_mean = np.mean(idh1_glioma.reshape(-1, 1), axis=0)
idh1_glioma_mean

[0.27337438 0.20391949 0.15090322 0.21191049 0.21416478 0.10835943
 0.10073197 0.20902753 0.20907219 0.39980268 0.2596621  0.13761726
 0.3719191  0.32927644 0.44465336 0.36366978 0.20586474 0.3051516
 0.15777864 0.17355087 0.15939042 0.1789848  0.34447303 0.20096543
 0.1005797  0.31148297 0.32306886 0.33107054 0.18298395 0.4940067
 0.4039505  0.25182098 0.15515463 0.2080874  0.5336286  0.30455193
 0.13861957 0.22318728 0.14053762 0.33787817 0.2490693  0.3004572
 0.2522408  0.22205468 0.12958615 0.17003375 0.15020294 0.20383433
 0.18883196 0.06894847 0.24010694 0.33457676 0.2055037  0.29667675
 0.1932096  0.25453883 0.2536336  0.20869689 0.14094712 0.36234003
 0.40188065 0.18325077 0.1575454  0.2154098  0.40619126 0.19585904
 0.38752145 0.37516633 0.13655064 0.28815016]
34
0.5336286


array([0.24754069])

In [21]:
prmt5_glioma = prmt5_scoring_matrix[:, -1]
prmt5_glioma = np.mean(prmt5_glioma.reshape(-1, 1), axis=0)
prmt5_glioma

array([0.14186052])

In [22]:
with open("/home/jienihu/sc/SLformer/data/saved_data/map/gene2id.pkl", 'rb') as f: 
    gene2id_map = pkl.load(f)
    
with open("/home/jienihu/sc/SLformer/data/saved_data/map/cancer_list.txt") as f:
    cancer_list = [line.rstrip('\n') for line in f]

# Create reverse mappings
id2gene_map = {i:g for g,i in gene2id_map.items()}
id2cancer_map = {i:cancer for i,cancer in enumerate(cancer_list)}
cancer2id_map = {cancer:i for i,cancer in enumerate(cancer_list)}

print(f"Loaded {len(gene2id_map)} genes and {len(cancer_list)} cancer types")
print(f"Cancer types: {cancer_list}")

Loaded 13459 genes and 9 cancer types
Cancer types: ['KIRC', 'COAD', 'LAML', 'OV', 'BRCA', 'CESC', 'SKCM', 'LUAD', 'Glioma']


In [23]:
go_dag = GODag("/home/jienihu/sc/SLformer/data/GO/go-basic.obo")
annotations = read_gaf("/home/jienihu/sc/SLformer/data/GO/goa_human.gaf", go_dag=go_dag)

# Load ID mapping from UniProt to gene symbols
id_mapping = pd.read_csv("/home/jienihu/sc/SLformer/data/GO/idmapping_2024_11_09.tsv", sep='\t')
id_mapping = dict(zip(id_mapping['From'], id_mapping['To']))

# Map annotations to gene symbols
anno_mapped = {}
for k, v in annotations.items():
    if k in id_mapping:
        anno_mapped[id_mapping[k]] = v

print(f"Loaded GO annotations for {len(anno_mapped)} genes")
print(f"Total GO terms in ontology: {len(go_dag)}")

/home/jienihu/sc/SLformer/data/GO/go-basic.obo: fmt(1.2) rel(2024-06-17) 45,494 Terms
HMS:0:00:15.300745 707,170 annotations READ: /home/jienihu/sc/SLformer/data/GO/goa_human.gaf 
36129 IDs in loaded association branch, BP
Loaded GO annotations for 17575 genes
Total GO terms in ontology: 45494


In [24]:
def get_go_term_details(go_terms):
    """Get detailed information about GO terms."""
    details = []
    for term_id in go_terms:
        if term_id in go_dag:
            term = go_dag[term_id]
            details.append({
                'id': term_id,
                'name': term.name,
                'namespace': term.namespace,
                'depth': term.depth
            })
    return details

def find_shared_goterms(gene1, gene2, return_terms=False):
    """Find shared GO terms between two genes."""
    go_terms_gene1 = {go_id for go_id in anno_mapped.get(gene1, [])}
    go_terms_gene2 = {go_id for go_id in anno_mapped.get(gene2, [])}
    
    shared_go_terms = go_terms_gene1.intersection(go_terms_gene2)
    gene1_terms = [terms['name'] for terms in get_go_term_details(go_terms_gene1)]
    gene2_terms = [terms['name'] for terms in get_go_term_details(go_terms_gene2)]
    shared_go_terms = [terms['name'] for terms in get_go_term_details(shared_go_terms)]

    if return_terms:
        return len(shared_go_terms), shared_go_terms, gene1_terms, gene2_terms
    else:
        return len(shared_go_terms)
    
def get_gene_sentence_neighbors(gene, cancer, common_data, top_n=10):
    """Get top co-expressed neighbors from gene sentence."""
    gene2id_map = common_data["gene2id_map"]
    gene2sent_map = common_data["gene2sent_map"]
    cancer2id_map = common_data["cancer2id_map"]
    
    if gene not in gene2id_map:
        return []
    
    gene_id = gene2id_map[gene]
    cancer_id = cancer2id_map[cancer]
    
    if cancer_id not in gene2sent_map or gene_id not in gene2sent_map[cancer_id]:
        return []
    
    # Get gene sentence (list of transformed gene IDs)
    gene_sentence = gene2sent_map[cancer_id][gene_id]
    
    # Convert back to gene names
    id2gene_map = {i:g for g,i in gene2id_map.items()}
    neighbors = []
    ngene = len(gene2id_map)  # Total number of genes
    
    for gene_idx in gene_sentence[1:top_n+1]:  # Skip first gene (itself)
        if gene_idx > 0:  # Skip padding tokens
            # Reverse the transformation: (g+1)+cancer*(ngene+1)
            # So: original_gene_id = (gene_idx - cancer_id * (ngene + 1)) - 1
            original_gene_id = (gene_idx - cancer_id * (ngene + 1)) - 1
            
            if original_gene_id in id2gene_map:
                neighbors.append(id2gene_map[original_gene_id])
            else:
                print(f"  Gene ID {gene_idx} -> {original_gene_id} not found in id2gene_map")
    
    return neighbors


In [25]:
gene_preds = pd.DataFrame()
for i in range(5):
    path = f'data/all_SL/pred_mix_slformer_kg_cv{i+1}.csv'
    df = pd.read_csv(path)
    gene_preds = pd.concat([gene_preds, df], ignore_index=True)

def search_pred_score(gene_pair, context=None):
    if context:
        df = gene_preds[((gene_preds['primary_gene'] == gene_pair[0]) & (gene_preds['partner_gene'] == gene_pair[1]) | (gene_preds['primary_gene'] == gene_pair[1]) & (gene_preds['partner_gene'] == gene_pair[0])) & (gene_preds['cancer'] == context)]
    else:
        df = gene_preds[((gene_preds['primary_gene'] == gene_pair[0]) & (gene_preds['partner_gene'] == gene_pair[1]) | (gene_preds['primary_gene'] == gene_pair[1]) & (gene_preds['partner_gene'] == gene_pair[0]))]
    if df.empty:
        print("end= No prediction found for the gene pair:", gene_pair)
        return 
    return df['score'].to_list() if not context else df['score'].to_list()[0]

In [26]:
IDH1_EMB_PATH = '/home/jienihu/sc/SLformer/experiment/emb/IDH1_PRKDC_emb'
BEST_FOLD = 3
idh1_embs = pd.DataFrame()
scoring_df = pd.read_csv(IDH1_PATH + idh1_res[np.argmax(idh1_glioma)])
score = scoring_df[f'score_{BEST_FOLD}']


contexts_avail = os.listdir(IDH1_EMB_PATH)
for context in contexts_avail:
    with_context_path = IDH1_EMB_PATH + f'/{context}'
    emb_file = os.listdir(with_context_path)[2 * np.argmax(idh1_glioma)]
    emb = pkl.load(open(with_context_path + f'/{emb_file}', 'rb'))
    idh1_emb, prkdc_emb = emb[BEST_FOLD-1][0], emb[BEST_FOLD-1][1]
    pair_emb = np.concatenate([idh1_emb, prkdc_emb], axis=0)
    idh1_embs = pd.concat([idh1_embs, pd.DataFrame(pair_emb).T], ignore_index=True)
    
idh1_embs['context'] = contexts_avail
idh1_embs['SLscore'] = score
idh1_embs.columns = [f'primary_emb_{i}' for i in range(idh1_emb.shape[0])] + [f'partner_emb_{i}' for i in range(prkdc_emb.shape[0])] + ['context', 'SLscore']
idh1_embs

,primary_emb_0,primary_emb_1,primary_emb_2,primary_emb_3,primary_emb_4,primary_emb_5,primary_emb_6,primary_emb_7,primary_emb_8,primary_emb_9,...,partner_emb_504,partner_emb_505,partner_emb_506,partner_emb_507,partner_emb_508,partner_emb_509,partner_emb_510,partner_emb_511,context,SLscore
0,0.165571,0.000000,0.097131,0.152494,0.090950,0.096055,0.161412,0.205624,0.089460,0.098780,...,0.057604,0.012493,0.251751,0.384084,0.000000,0.000000,0.103857,0.347548,COAD,0.012469
1,0.025391,0.000000,0.000000,0.020430,0.078672,0.198429,0.412788,0.314775,0.030255,0.000092,...,0.281317,0.003405,0.178923,0.277670,0.049380,0.118358,0.000000,0.384764,LAML,0.084339
2,0.094682,0.013012,0.284993,0.004009,0.077009,0.076318,0.352992,0.038492,0.380259,0.059880,...,0.052812,0.000000,0.393915,0.259869,0.102362,0.123124,0.005940,0.775608,OV,0.268113
3,0.000000,0.000000,0.149566,0.245552,0.029750,0.133357,0.089524,0.128748,0.026488,0.244730,...,0.198616,0.001938,0.365141,0.473631,0.454249,0.004700,0.091806,0.232266,BRCA,0.435550
4,0.191499,0.003051,0.072826,0.080107,0.210498,0.242653,0.283781,0.081931,0.128906,0.009789,...,0.151785,0.006049,0.341714,0.067987,0.007269,0.150766,0.007905,0.631405,CESC,0.002941
5,0.026961,0.000000,0.000000,0.002221,0.001730,0.041531,0.074549,0.118929,0.117865,0.503274,...,0.222365,0.000000,0.254017,0.011614,0.189765,0.226391,0.003208,0.043716,SKCM,0.020995
6,0.055962,0.000000,0.067023,0.068055,0.067863,0.220504,0.083298,0.086192,0.022402,0.202265,...,0.235961,0.004871,0.263633,0.023978,0.198554,0.004168,0.000272,0.258095,LUAD,0.417944
7,0.059640,0.000000,0.226064,0.043419,0.002492,0.277073,0.197727,0.111201,0.079738,0.129580,...,0.083635,0.000000,0.263663,0.368031,0.303156,0.009088,0.002110,0.005268,Glioma,0.903764


In [27]:
gene_pair = ['PARP1', 'BRCA1']
gene_pair_map = {
    'primary': 'BRCA1',
    'partner': 'PARP1'
}
context = 'BRCA'
score = search_pred_score(gene_pair, context=context)
# ===============================================================

# gene_pair = ['KRAS', 'TBK1']
# gene_pair_map = {
#     'primary': 'KRAS',
#     'partner': 'TBK1'
# }
# context = 'LUAD'
# score = search_pred_score(gene_pair, context=context)

# ===============================================================

# gene_pair = ['IDH1', 'PRKDC']
# gene_pair_map = {
#     'primary': 'IDH1',
#     'partner': 'PRKDC'
# }
# context = 'Glioma'
# score = idh1_embs['SLscore']

# ===============================================================

# gene_pair = ['PRMT5', 'MAT2A']
# gene_pair_map = {
#     'primary': 'MAT2A',
#     'partner': 'PRMT5'
# }
# context = 'Glioma'
# score = prmt5_glioma[0]

# ===============================================================

# gene_pair = ['BRCA2', 'EZH2']
# gene_pair_map = {
#     'primary': gene_pair[0],
#     'partner': gene_pair[1]
# }
# context = 'BRCA'
# score = search_pred_score(gene_pair, context=context)

# ===============================================================

# gene_pair = ['ERH', 'KRAS']
# gene_pair_map = {
#     'primary': gene_pair[0],
#     'partner': gene_pair[1]
# }
# context = 'LUAD'
# score = search_pred_score(gene_pair, context=context)
# ===============================================================

# gene_pair = ['GATA2', 'KRAS']
# gene_pair_map = {
#     'primary': gene_pair[0],
#     'partner': gene_pair[1]
# }
# context = 'LUAD'
# score = search_pred_score(gene_pair, context=context)
# ===============================================================

# gene_pair = ['KRAS', 'PLK1']
# gene_pair_map = {
#     'primary': gene_pair[0],
#     'partner': gene_pair[1]
# }
# context = 'LUAD'
# score = search_pred_score(gene_pair, context=context)

In [28]:
def get_common_data(sent_n=200):

    common_data_path = {
        'geneformer_emb_map': '/home/jienihu/sc/SLformer/data/saved_data/map/geneformer_emb.pkl', 
        'geneformer_emb_mtx': '/home/jienihu/sc/SLformer/data/saved_data/emb/geneformer_emb.npy', 
        "gene2sent_map": f"/home/jienihu/sc/SLformer/data/saved_data/map/gene2sent_n{sent_n}.pkl",
        "sent_mask_map": f"/home/jienihu/sc/SLformer/data/saved_data/map/sent_mask_n{sent_n}.pkl",
        "gene2id_map": "/home/jienihu/sc/SLformer/data/saved_data/map/gene2id.pkl",
    }
    
    # print(common_data_path)
    for data, path in common_data_path.items():
        if not os.path.exists(path):
            raise Exception(f"{data,path} cannot be found, please first construct it.")
    
    common_data = {}
    for data in ["geneformer_emb_map","gene2sent_map","sent_mask_map","gene2id_map"]:
        with open(common_data_path[data], 'rb') as f:
            common_data[data] = pkl.load(f)
    common_data["geneformer_emb_mtx"] = np.load(common_data_path["geneformer_emb_mtx"])
     
    cancer2id_map = {c:i for i,c in enumerate(cancer_list)}
    common_data["cancer_list"] = cancer_list
    common_data["cancer2id_map"] = cancer2id_map
    go_anno = pd.read_csv(os.path.join("/home/jienihu/sc/SLformer/data", "GO", "go_anno_popular.csv"))
    gene2go_map = dict(zip(go_anno["gene_id"], go_anno["GO_id"]))
    # gene2go_map = dict(zip(go_anno["gene_id"], go_anno["annotation"]))
    common_data["gene2go_map"] = gene2go_map
    
    # Load co-expression data
    common_data["cos_sim_kg"] = pd.read_csv("/home/guoyu/SLformer_interpretation/data/all_SL/cos_sim_kg_sorted.csv")
    common_data["func_cos_sim_kg"] = pd.read_csv("/home/guoyu/SLformer_interpretation/data/all_SL/func_cos_sim_kg.csv")
    
    return common_data
common_data = get_common_data()

In [29]:
def load_embedding_features(gene_pair_map, embedding_base_path="embedding_saved"):
    """Load important embedding features for a gene pair across all contexts."""
    
    primary_gene = gene_pair_map['primary']
    partner_gene = gene_pair_map['partner']
    
    # Choose embedding type directory based on USE_GF flag
    embedding_type_candidates = ["crossemb_important", "crossemb-important"]

    embedding_type_dir = embedding_type_candidates[0]
    for d in embedding_type_candidates:
        if os.path.exists(os.path.join(embedding_base_path, d)):
            embedding_type_dir = d
            break
    
    # Construct the directory path for this gene pair
    gene_pair_dir = f"{primary_gene}-{partner_gene}"
    full_path = os.path.join(embedding_base_path, embedding_type_dir, gene_pair_dir)
    
    if not os.path.exists(full_path):
        # Try reverse order
        gene_pair_dir = f"{partner_gene}-{primary_gene}"
        full_path = os.path.join(embedding_base_path, embedding_type_dir, gene_pair_dir)
        print(full_path)
        if not os.path.exists(full_path):
            print(f"Warning: No embedding features found for {primary_gene}-{partner_gene} using cross-attention embeddings")
            return None, None
    
    # Load primary and partner embedding features
    primary_features_path = os.path.join(full_path, "primary_important_features.csv")
    partner_features_path = os.path.join(full_path, "partner_important_features.csv")
    
    primary_features = None
    partner_features = None
    
    primary_features = pd.read_csv(primary_features_path)
    partner_features = pd.read_csv(partner_features_path)

    
    return primary_features, partner_features

def format_embedding_features(features_df, gene_type="primary"):
    """Format embedding features for prompt inclusion."""
    if features_df is None:
        return f"No {gene_type} embedding features available."
    
    context_data = []
    for _, row in features_df.iterrows():
        cancer = row['cancer']
        score = row['score']
        
        # Get non-zero embedding features
        emb_cols = [col for col in features_df.columns if col.startswith(f'{gene_type}_emb_')]
        active_features = []
        for col in emb_cols:
            if row[col] > 0:
                feature_idx = col.split('_')[-1]
                active_features.append(f"emb_{feature_idx}: {row[col]:.3f}")
        
        context_info = f"  - {cancer} (SL-Score: {score:.3f}): {', '.join(active_features) if active_features else 'No active features'}"
        context_data.append(context_info)
    
    return f"{gene_type.capitalize()} Gene Embedding Features:\n" + "\n".join(context_data)

def _analyze_embedding_contexts(features, gene_type):
    """Helper function to generate per-gene embedding analysis with cross-context comparison."""
    analysis = []
    cancer_types = features['cancer'].unique()
    
    for cancer in cancer_types:
        subset = features[features['cancer'] == cancer]
        score = subset['score'].values[0]
        
        # Analyze activation patterns for this context
        emb_cols = [col for col in features.columns if col.startswith(f'{gene_type}_emb_')]
        row = subset.iloc[0]
        
        # Identify top dimensions and their activation levels
        top_dims = sorted([(col, row[col]) for col in emb_cols], key=lambda x: -x[1])[:5]
        top_dims_str = ', '.join([f"dim_{col.split('_')[-1]}({val:.3f})" for col, val in top_dims])
        
        # Compare to average activation across all contexts
        avg_activations = features[emb_cols].mean().sort_values(ascending=False)
        context_specific = []
        for col, val in top_dims:
            dim_idx = col.split('_')[-1]
            avg_val = avg_activations[col]
            if val > 1.5*avg_val:  # Arbitrarily defined as context-specific
                context_specific.append(f"dim_{dim_idx}(x{val/avg_val:.1f} vs avg)")
        
        context_specific_str = f"Context-specific: {', '.join(context_specific)}" if context_specific else "No strongly context-specific dimensions"
        
        analysis.append(f"  {cancer}: Top dimensions - {top_dims_str} | {context_specific_str}")

    
    return analysis

In [30]:


_pair_genes = [gene_pair_map.get('primary'), gene_pair_map.get('partner')]
import re
import gseapy as gp
def _clean_enrichment_terms(terms):
    """Lightly normalize enrichment term strings for better semantic matching.
    - remove species fragments
    - replace underscores
    - strip parentheses-only identifiers
    - lowercase for stability
    """
    cleaned = []
    pat_species = re.compile(r"\s*-?\s*Homo sapiens\b|\s*\(Homo sapiens\)\s*", re.IGNORECASE)
    pat_go_id = re.compile(r"\s*\(GO:\d+\)\s*", re.IGNORECASE)
    for t in terms:
        s = str(t)
        s = pat_species.sub("", s)
        s = pat_go_id.sub("", s)
        s = s.replace("_", " ")
        s = re.sub(r"\s+", " ", s).strip()
        cleaned.append(s.lower())
    return cleaned

def compute_enrichment_text_for_genes(genes, gene_sets=("Reactome_2022", "KEGG_2021_Human", "GO_Biological_Process_2021"), top_k=10):
    """Query multiple libraries with Enrichr for the provided genes and build a concise text.
    Returns a comma-joined string of the top terms (deduplicated, cleaned) across libraries.
    """
    if not genes:
        return ""
    agg_terms = []
    seen = set()
    # Try each library; keep up to top_k unique terms total
    for gs in gene_sets:
        try:
            enr = gp.enrichr(
                gene_list=list(genes),
                gene_sets=[gs],
                organism='Human',
                outdir=None,
                cutoff=1.0,
            )
            df = enr.results
            if df is None or df.empty:
                continue
            sort_col = "Adjusted P-value" if "Adjusted P-value" in df.columns else ("P-value" if "P-value" in df.columns else None)
            if sort_col is not None:
                df = df.sort_values(sort_col, ascending=True)
            raw_terms = df["Term"].astype(str).tolist()
            for term in _clean_enrichment_terms(raw_terms):
                if term not in seen:
                    agg_terms.append(term)
                    seen.add(term)
                if len(agg_terms) >= top_k:
                    break
            if len(agg_terms) >= top_k:
                break
        except Exception:
            continue
    return ", ".join(agg_terms)

enrichment_text = compute_enrichment_text_for_genes(
    _pair_genes,
    gene_sets=("Reactome_2022", "KEGG_2021_Human", "GO_Biological_Process_2021",
               "GO_Molecular_Function_2021", "GO_Cellular_Component_2021"),
    top_k=20,
)

In [31]:
enrichment_text

'sumoylation of dna damage response and repair proteins r-hsa-3108214, homology directed repair r-hsa-5693538, dna double-strand break repair r-hsa-5693532, sumo e3 ligases sumoylate target proteins r-hsa-3108232, sumoylation r-hsa-2990846, dna repair r-hsa-73894, polb-dependent long patch base excision repair r-hsa-110362, hdr thru mmej (alt-nhej) r-hsa-5685939, impaired brca2 binding to palb2 r-hsa-9709603, resolution of ap sites via multiple-nucleotide patch replacement pathway r-hsa-110373, defective hdr thru homologous recombination (hrr) due to brca1 loss-of-function r-hsa-9701192, resolution of d-loop structures thru synthesis-dependent strand annealing (sdsa) r-hsa-5693554, downregulation of smad2/3:smad4 transcriptional activity r-hsa-2173795, metalloprotease dubs r-hsa-5689901, resolution of d-loop structures thru holliday junction intermediates r-hsa-5693568, transcriptional regulation by e2f6 r-hsa-8953750, resolution of d-loop structures r-hsa-5693537, impaired brca2 bindi

In [32]:
# Soft, data-driven topic nudging toward metabolism and redox cofactors
from typing import Tuple, List

METABOLIC_KEYWORDS = [
    # general
    "metabol", "metabolic process", "biosynthesis", "anabolic", "catabolic",
    # carbon/energy
    "glycolysis", "glycolytic", "gluconeogenesis", "tricarboxylic", "tca", "krebs",
    "oxidative phosphorylation", "oxphos", "electron transport", "warburg",
    "pentose phosphate", "ppp",
    # lipids
    "beta oxidation", "fatty acid", "lipid", "cholesterol", "sterol", "isoprenoid", "mevalonate",
    # one-carbon / nucleotides
    "one-carbon", "folate", "methionine", "s-adenosylmethionine", "sam",
    "nucleotide", "pyrimidine", "purine", "de novo",
    # amino acids / redox
    "amino acid", "glutamine", "glutaminolysis", "glutathione", "redox", "reactive oxygen", "ros",
    # cofactors (general mentions counted under metabolism as well)
    "nadph", "nadp", "nadp+", "nadh", "nad+", "flavin", "fmn", "fad"
]

REDOX_KEYWORDS = [
    # cofactor balance and producers/consumers
    "nadph", "nadp", "nadp+", "nad+/nadh", "nadh", "nad+",
    "pentose phosphate", "g6pd", "glucose-6-phosphate dehydrogenase",
    "6-phosphogluconate dehydrogenase", "pgd",
    "malic enzyme", "me1", "me2",
    "isocitrate dehydrogenase", "idh1", "idh2",
    "thioredoxin", "glutathione", "gpx", "gr", "txnr",
    "ferroptosis", "lipid peroxidation", "oxidative stress",
]

def estimate_metabolic_support(enrichment_text: str) -> Tuple[float, List[str]]:
    """Estimate metabolic-topic support from a comma-joined enrichment summary.
    Returns (support_ratio, representative_terms).
    """
    if not enrichment_text:
        return 0.0, []
    terms = [t.strip().lower() for t in enrichment_text.split(',') if t.strip()]
    if not terms:
        return 0.0, []
    matched = []
    for t in terms:
        if any(kw in t for kw in METABOLIC_KEYWORDS):
            matched.append(t)
    support = len(matched) / max(len(terms), 1)
    # return up to 5 representative matched terms in original order
    seen = set()
    reps = []
    for t in matched:
        if t not in seen:
            reps.append(t)
            seen.add(t)
        if len(reps) >= 5:
            break
    return support, reps


def estimate_redox_support(enrichment_text: str) -> Tuple[float, List[str]]:
    """Estimate redox/NAD(P)H cofactor-related support from enrichment terms.
    Returns (support_ratio, representative_terms).
    """
    if not enrichment_text:
        return 0.0, []
    terms = [t.strip().lower() for t in enrichment_text.split(',') if t.strip()]
    if not terms:
        return 0.0, []
    matched = []
    for t in terms:
        if any(kw in t for kw in REDOX_KEYWORDS):
            matched.append(t)
    support = len(matched) / max(len(terms), 1)
    seen = set()
    reps = []
    for t in matched:
        if t not in seen:
            reps.append(t)
            seen.add(t)
        if len(reps) >= 5:
            break
    return support, reps

In [ ]:
from pathlib import Path

PROMPT_TEMPLATE_PATH = Path('/home/guoyu/SLformer_interpretation/src/LLM_interpretation/prompt_template.txt')

def generate_contextual_embedding_interpretation_prompt(gene_pair_map, context, embedding_base_path="embedding_saved", enrichment_text=None, metabolic_soft_prior: bool = True):
    """Generate a comprehensive prompt for interpreting SLformer embeddings with cross-context analysis and biological mechanism deduction."""

    embedding_base_path = Path(embedding_base_path)
    if not embedding_base_path.is_absolute():
        candidate_paths = [
            Path('/home/guoyu/SLformer_interpretation/src/LLM_interpretation') / embedding_base_path,
            Path('/home/guoyu/SLformer_interpretation') / embedding_base_path,
        ]
        for candidate in candidate_paths:
            if candidate.exists():
                embedding_base_path = candidate
                break

    _, overlaps, primary_terms, partner_terms = find_shared_goterms(
        gene_pair_map['primary'], gene_pair_map['partner'], return_terms=True
    )
    primary_features, partner_features = load_embedding_features(gene_pair_map, str(embedding_base_path))

    if primary_features is None and partner_features is None:
        return f"Error: No embedding features found for gene pair {gene_pair_map['primary']}-{gene_pair_map['partner']} using cross-attention embeddings"

    analysis = []
    if primary_features is not None and not primary_features.empty:
        analysis.append(f"Primary Gene ({gene_pair_map['primary']}) Contextual Embedding Analysis:")
        analysis.extend(_analyze_embedding_contexts(primary_features, 'primary'))
    if partner_features is not None and not partner_features.empty:
        analysis.append(f"\nPartner Gene ({gene_pair_map['partner']}) Contextual Embedding Analysis:")
        analysis.extend(_analyze_embedding_contexts(partner_features, 'partner'))
    

    embedding_type_str = "cross-attention"
    available_contexts = []
    if primary_features is not None and not primary_features.empty:
        available_contexts.extend(primary_features['cancer'].tolist())
    if partner_features is not None and not partner_features.empty:
        available_contexts.extend(partner_features['cancer'].tolist())

    if enrichment_text:
        topic_profile_line = (
            "Context-enrichment profile: the available terms emphasize DNA repair and nonhomologous end joining, DNA double-strand break repair, cellular stress responses, metabolism-related processes, protein ubiquitination, and immune/stress signaling; use these as supporting context rather than as standalone proof."
        )
    else:
        topic_profile_line = ""

    mech_ranking_hint = (
        "If several mechanism families are plausible, rank them by embedding support and carry the best-supported family through the downstream chain from molecular state to pathway dependency to stress or fate consequence. "
        "The top-ranked chain should explicitly traverse the requested levels instead of stopping at a pathway label."
    )
    metabolic_note = (
        "If the evidence supports a metabolic axis, explain how the pathway shift could propagate into energy, redox, or other stress imbalance before moving to cell fate."
    ) if metabolic_soft_prior else ""
    redox_note = (
        "If a redox axis is supported, connect it to DNA damage burden or repair dependence only when the embeddings and enrichment context support that bridge."
    ) if metabolic_soft_prior else ""
    guardrail_line = (
        "Keep the explanation abstract and reusable across gene pairs; do not name a specific mutation, metabolite, pathway node, or drug unless the embeddings plus GO context directly justify it."
    )
    depth_guidance = (
        "Use a layered mechanistic ladder when the evidence supports it:\n"
        "- Molecular level: identify an upstream state change implied by the embeddings and GO context, but keep it abstract unless a specific alteration is directly supported.\n"
        "- Cellular pathway level: explain which dependency, repair choice, or signaling branch becomes stronger or weaker.\n"
        "- Metabolic level: connect the pathway shift to energy, redox, or other stress consequences only if the evidence supports that move.\n"
        "- Cell-fate level: connect the stress or dependency shift to checkpoint activation, arrest, apoptosis, or failure of a protective response.\n"
        "- Therapeutic level: state how a second perturbation, inhibitor, or stressor could synergize, sensitize, or partially rescue the phenotype.\n"
        "If only part of the ladder is supported, say so and do not force the remaining layers."
    )

    available_contexts_str = ', '.join(sorted(set(available_contexts))) if available_contexts else 'Unknown'
    primary_terms_str = ", ".join(primary_terms) if primary_terms else "N/A"
    partner_terms_str = ", ".join(partner_terms) if partner_terms else "N/A"
    overlaps_str = ", ".join(overlaps) if overlaps else "N/A"
    analysis_str = "\n".join(analysis)

    template = PROMPT_TEMPLATE_PATH.read_text(encoding='utf-8')
    query_prompt = template.format(
        embedding_type_str=embedding_type_str,
        gene_primary=gene_pair_map['primary'],
        gene_partner=gene_pair_map['partner'],
        primary_terms_str=primary_terms_str,
        partner_terms_str=partner_terms_str,
        overlaps_str=overlaps_str,
        analysis_str=analysis_str,
        target_context=context,
        available_contexts_str=available_contexts_str,
        topic_profile_line=topic_profile_line,
        depth_guidance=depth_guidance,
        mech_ranking_hint=mech_ranking_hint,
        metabolic_note=metabolic_note,
        redox_note=redox_note,
        guardrail_line=guardrail_line,
    )
    return query_prompt.strip()

In [34]:
print(generate_contextual_embedding_interpretation_prompt(
    gene_pair_map, context, embedding_base_path="embedding_saved", enrichment_text=enrichment_text
))

You are a computational biologist specialized in mechanistic interpretation of deep learning models for synthetic lethality prediction.
Your task is to analyze context-aware embeddings from cross-attention representations, identifying patterns that correlate with cancer-specific biological mechanisms. These embeddings represent learned features distinguishing synthetic lethal relationships in specific cancer contexts. Your analysis should focus on the differential activation patterns across cancer types to deduce context-specific biological mechanisms.

**Anti-anchoring and evidence discipline:**
- Do not anchor on the most famous or canonical function of either gene if the embedding evidence points elsewhere.
- If multiple plausible mechanism axes are supported, keep them separate unless the prompt evidence explicitly links them.
- Distinguish what is directly supported by embedding evidence, what is supported by GO/enrichment context, and what remains hypothetical.
- Include one conc

In [35]:
# MedCPT and MPNet text similarity utilities
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModel
import torch, torch.nn.functional as F


# Device and model paths
_device = "cuda:2" if torch.cuda.is_available() else "cpu"
_medcpt_path = "/data/guoyu/HF-models/MedCPT-Query-Encoder"
_mpnet_path = "/data/guoyu/HF-models/all-mpnet-base-v2"
_bge_path = '/data/guoyu/HF-models/bge-large-en-v1.5'

# Initialize MedCPT via transformers (use [CLS] of last hidden state)
medcpt_tokenizer = AutoTokenizer.from_pretrained(_medcpt_path, local_files_only=True)
medcpt_model = AutoModel.from_pretrained(_medcpt_path, local_files_only=True)
medcpt_model.to(_device)
medcpt_model.eval()

# Initialize MPNet via SentenceTransformers (uses mean pooling by default)
mpnet_model = SentenceTransformer(_mpnet_path, device=_device, local_files_only=True)
bge_model = SentenceTransformer(_bge_path, device=_device, local_files_only=True)




def _medcpt_encode(texts, max_length=64):
    """Encode texts with MedCPT (transformers) using CLS token and L2 normalization."""
    if not texts:
        return None
    enc = medcpt_tokenizer(
        texts,
        truncation=True,
        padding=True,
        return_tensors='pt',
        max_length=max_length,
    )
    enc = {k: v.to(_device) for k, v in enc.items()}
    with torch.no_grad():
        cls = medcpt_model(**enc).last_hidden_state[:, 0, :]
        cls = F.normalize(cls, p=2, dim=1)
    return cls


def medcpt_text_similarity(text_a, text_b, max_length=64):
    """Cosine similarity using MedCPT (CLS pooling via transformers)."""
    if not text_a or not text_b:
        return float('nan')
    embs = _medcpt_encode([text_a, text_b], max_length=max_length)
    if embs is None or embs.shape[0] < 2:
        return float('nan')
    # cosine of L2-normalized vectors equals dot product
    return float(torch.sum(embs[0] * embs[1]).item())


def mpnet_text_similarity(text_a, text_b):
    """Cosine similarity using MPNet (SentenceTransformers)."""
    if not text_a or not text_b:
        return float('nan')
    embs = mpnet_model.encode([text_a, text_b], normalize_embeddings=True)
    return float(util.cos_sim(embs[0], embs[1]))

def bge_text_similarity(text_a, text_b):
    if not text_a or not text_b:
        return float('nan')
    embs = bge_model.encode([text_a, text_b], normalize_embeddings=True)
    return float(util.cos_sim(embs[0], embs[1]))


/home/guoyu/SLformer_interpretation/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from pathlib import Path
import sys
import importlib
import os

LLM_PART_DIR = (Path.cwd() / 'src' / 'LLM_part').resolve()
if str(LLM_PART_DIR) not in sys.path:
    sys.path.insert(0, str(LLM_PART_DIR))

# Reload prompt templates so edits to prompt_api/strategy_prompts.py take effect
import prompt_api.strategy_prompts as strategy_prompts
importlib.reload(strategy_prompts)
import prompt_api.strategies as strategies
importlib.reload(strategies)

from prompt_api.client import AigcBestChatClient

client = AigcBestChatClient()

# The previous run was truncating the wet-lab validation sentence because the
# final generation was capped too tightly. Give the model more room and make the
# prompt explicitly require a complete validation paragraph.
GENERATION_MAX_TOKENS = 10000
STRATEGY_MAX_TOKENS = GENERATION_MAX_TOKENS // 3


def run_single_prompt_with_strategies() -> dict:

    prompt = generate_contextual_embedding_interpretation_prompt(
        gene_pair_map,
        context,
        embedding_base_path=str(LLM_PART_DIR / 'embedding_saved'),
        enrichment_text=enrichment_text,
    )

    draft = client.complete(prompt, temperature=0.7, top_p=0.95, max_tokens=GENERATION_MAX_TOKENS).text

    # CoVe
    trace_cove = strategies.run_cove(
        client,
        prompt,
        draft=draft,
        n_questions=5,
        temperature=0.7,
        top_p=0.9,
        max_tokens=STRATEGY_MAX_TOKENS,
    )
    cove_questions = trace_cove.questions.text
    cove_answers = trace_cove.answers.text
    cove_result = trace_cove.final.text if trace_cove.final else trace_cove.initial.text

    # Self-refine (1 round)
    trace_refine = strategies.run_self_refine(
        client,
        prompt,
        draft=cove_result,
        rounds=1,
        temperature=0.7,
        top_p=0.9,
        max_tokens=STRATEGY_MAX_TOKENS,
    )
    self_refine_feedback = trace_refine.feedback.text
    self_refine_result = trace_refine.final.text if trace_refine.final else trace_refine.initial.text

    return {
        'prompt': prompt,
        'draft': draft,
        'cove_questions': cove_questions,
        'cove_answers': cove_answers,
        'cove_result': cove_result,
        'self_refine_feedback': self_refine_feedback,
        'self_refine_result': self_refine_result,
    }

print(f"============== Model: {client.model} ==============")
run = run_single_prompt_with_strategies()


def format_trace_report(run: dict) -> str:
    # Match the compact report style used by the KRAS-pair agent outputs.
    lines = []
    lines.append(f"PAIR: {gene_pair[0]}-{gene_pair[1]}")
    lines.append(f"TARGET_CONTEXT: {context}")
    lines.append('')
    lines.append('=== GENERATED (API; CoVe + self-refine) ===')
    lines.append(run['self_refine_result'])
    lines.append('')
    lines.append('=== TRACE (for debugging) ===')
    lines.append('[CoVe questions]')
    lines.append(run['cove_questions'])
    lines.append('')
    lines.append('[CoVe answers]')
    lines.append(run['cove_answers'])
    lines.append('')
    lines.append('[Self-refine feedback]')
    feedback_lines = [line for line in run['self_refine_feedback'].splitlines() if line.strip()]
    lines.append('\n'.join(feedback_lines[:6]))
    lines.append('')
    lines.append('NOTE: No ground-truth feature coverage metrics are included for this pair.')
    return '\n'.join(lines)


OUT_DIR = Path('/home/guoyu/SLformer_interpretation/output/explanation_agent_output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

report_path = OUT_DIR / f"{gene_pair[0]}-{gene_pair[1]}_{context.lower()}_{client.model.lower()}.txt"
report_path.write_text(format_trace_report(run).strip() + '\n', encoding='utf-8')

print('Saved CoVe + self-refine report to:', report_path)

============== Model: gpt-5.4 ==============
Saved CoVe + self-refine report to: /home/guoyu/SLformer_interpretation/src/LLM_part/explanation_agent_output/PARP1-BRCA1_brca_gpt-5.4.txt
